# Análisis: preguntas de negocio y arquetipos de clientes

Este notebook usa las tablas limpias generadas por 01_limpieza.ipynb.

## 1. Carga de datos limpios

In [1]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent
PROC = RAIZ / "data" / "processed"
FIGS = RAIZ / "reports" / "figures"
DATA = RAIZ / "data"

archivos = ["customers", "orders", "order_items", "products"]
faltantes = [a for a in archivos if not (PROC / f"{a}.csv").exists()]
assert not faltantes, f"Faltan archivos en {PROC}: {faltantes}. Ejecuta primero 01_limpieza.ipynb"

customers   = pd.read_csv(PROC / "customers.csv",
                          parse_dates=["fecha_registro", "fecha_nacimiento", "primera_compra", "cliente_desde"])
orders      = pd.read_csv(PROC / "orders.csv", parse_dates=["fecha"])
order_items = pd.read_csv(PROC / "order_items.csv")
products    = pd.read_csv(PROC / "products.csv", parse_dates=["fecha_lanzamiento"])
stores = pd.read_csv(DATA / "stores.csv")  

FECHA_CORTE = pd.Timestamp("2026-09-30")

for nombre, df in {"customers": customers, "orders": orders, "order_items": order_items,
                   "products": products, "stores": stores}.items():
    print(f"{nombre}: {df.shape[0]} filas")

customers: 3005 filas
orders: 22651 filas
order_items: 47641 filas
products: 240 filas
stores: 5 filas


## 2. Tabla base de ventas

Se unen las líneas de venta con su orden, su producto y su cliente, y se calcula la venta neta de cada línea. Se aplican las definiciones de la limpieza:
- Solo cuentan como venta las órdenes completadas.
- Venta neta = cantidad × precio_unitario × (1 − descuento_pct / 100).
- El total de ventas de la marca incluye todas las ventas completadas; el análisis por cliente usa solo las órdenes atribuibles a un cliente registrado, sin cuentas internas.

In [3]:
ventas = (order_items
    .merge(orders, on="order_id", how="inner")
    .merge(products[["sku", "coleccion", "categoria", "fecha_lanzamiento"]], on="sku", how="left")
    .merge(customers[["customer_id", "cuenta_interna"]], on="customer_id", how="left")
    .merge(stores.rename(columns={"nombre": "tienda"}), on="tienda_id", how="left"))

ventas = ventas[ventas["es_venta"]].copy()
ventas["venta_neta"] = ventas["cantidad"] * ventas["precio_unitario"] * (1 - ventas["descuento_pct"] / 100)
ventas["cuenta_interna"] = ventas["cuenta_interna"].fillna(False).astype(bool)

# Ventas que se pueden asignar a un cliente real
ventas_cli = ventas[ventas["cliente_atribuible"] & ~ventas["cuenta_interna"]]

total_marca = ventas["venta_neta"].sum()
total_cli = ventas_cli["venta_neta"].sum()

print(f"Ventas netas de la marca: {total_marca:,.0f} MXN ({ventas['order_id'].nunique():,} órdenes)")
print(f"Ventas atribuibles a clientes: {total_cli:,.0f} MXN ({total_cli / total_marca:.1%})")
print(f"Clientes con al menos una compra: {ventas_cli['customer_id'].nunique():,}")

Ventas netas de la marca: 45,033,143 MXN (21,297 órdenes)
Ventas atribuibles a clientes: 42,247,804 MXN (93.8%)
Clientes con al menos una compra: 2,877


## 3. Pregunta 1: ¿quiénes se interesan por las nuevas colecciones?

### Verificación previa: ventas anteriores al lanzamiento del producto (P3)

Para medir el interés por lo nuevo hay que comparar la fecha de cada venta con la fecha de lanzamiento del producto. Antes se revisa que esa comparación tenga sentido: un producto no debería venderse antes de su lanzamiento.

In [7]:
ventas["dias_desde_lanz"] = (ventas["fecha"] - ventas["fecha_lanzamiento"]).dt.days
antes = ventas["dias_desde_lanz"] < 0

print("Líneas vendidas antes del lanzamiento:", antes.sum(), f"({antes.sum() / ventas['fecha_lanzamiento'].notna().sum():.2%})")
print("Clientes distintos:", ventas.loc[antes, "customer_id"].nunique())

ids_antes = ventas.loc[antes, "customer_id"].dropna().unique()
evento = customers.set_index("customer_id")["nota_evento_lanzamiento"]
print("De esos clientes, con nota de evento de lanzamiento:", int(evento.reindex(ids_antes).fillna(False).sum()), "de", int(evento.sum()))

(ventas[antes].groupby("coleccion")["dias_desde_lanz"]
   .agg(lineas="count", dias_min="min", dias_mediana="median", dias_max="max")
   .sort_values("lineas", ascending=False))

Líneas vendidas antes del lanzamiento: 1886 (5.84%)
Clientes distintos: 669
De esos clientes, con nota de evento de lanzamiento: 5 de 15


,lineas,dias_min,dias_mediana,dias_max
coleccion,,,,
PV24,1043,-45.0,-29.0,-1.0
OI26,291,-938.0,-403.0,-1.0
PV26,238,-770.0,-362.5,-1.0
OI25,151,-593.0,-240.0,-14.0
PV25,101,-379.0,-213.0,-1.0
OI24,62,-231.0,-108.5,-6.0


**Idea principal:** hay productos que aparecen vendidos antes de su fecha de lanzamiento, algo que no debería pasar. Son 1.886 líneas (5,8% de las ventas con colección), de 669 clientes. Esto significa que, para esas ventas, la colección que indica el catálogo no es confiable.

**Cómo leer la tabla:** dias_desde_lanz es la diferencia entre la fecha de la venta y la fecha de lanzamiento del producto. Un valor negativo significa que la venta ocurrió antes del lanzamiento.
- dias_min: la venta más lejana al lanzamiento (por ejemplo, −938 = vendida 938 días antes de que el producto existiera).
- dias_max: la venta más cercana al lanzamiento (−1 = vendida un día antes).
- dias_mediana: el valor típico; la mitad de las ventas anticipadas está por encima de ese valor y la otra mitad por debajo.

**Hay dos situaciones distintas:**

1. **PV24: 1.043 líneas, entre 1 y 45 días antes. Tiene explicación.** Los datos de órdenes empiezan el 2024-01-01, pero la primera colección del catálogo (PV24) se lanzó el 2024-02-15. Lo que se vendió en esas seis semanas eran productos de la temporada anterior, que no está en el catálogo; por eso aparecen con códigos de PV24.

2. **Resto de colecciones: 843 líneas, hasta 938 días antes. No tiene explicación de negocio.** Por ejemplo, hay productos de OI26 (lanzada en agosto de 2026) vendidos desde enero de 2024. Son casos aislados: las 291 líneas de OI26 están repartidas en más de dos años y equivalen a menos del 1% de lo que vendió la marca en ese periodo. Si fuera un comportamiento real, se verían ventas constantes de esos productos; en cambio, aparecen de forma esporádica. Lo más probable es un error en la fecha o en el SKU de esas órdenes.

**No es una preventa:** si fuera acceso anticipado para clientes especiales, serían pocos clientes y concentrados en las colecciones nuevas. En cambio, son 669 clientes, en todas las colecciones, y solo 5 de ellos asistieron a un evento de lanzamiento.

In [ ]:
ventas["coleccion_valida"] = ventas["coleccion"].notna() & (
    ventas["coleccion"].eq("BASICOS") | (ventas["dias_desde_lanz"] >= 0))

# ventas_cli se recalcula para que también tenga las columnas nuevas
ventas_cli = ventas[ventas["cliente_atribuible"] & ~ventas["cuenta_interna"]]

print("Líneas con colección válida:", ventas["coleccion_valida"].sum(), "de", len(ventas))

**Tratamiento:** se crea la bandera coleccion_valida. Las ventas anteriores al lanzamiento se mantienen en el total de ventas, pero no cuentan como compras de una colección.

**Impacto si no se corrige:** se contarían como compradores de una colección a clientes que compraron antes de que existiera. Además, todas las ventas de enero y febrero de 2024 se asignarían a PV24, inflando esa colección.